In [ ]:
"""
Checkpointing + Time Travel demo on the parallel research agent.

1. Run the agent (it pauses at human_review, we approve -> report)
2. INSPECT  - list every checkpoint of the run
3. REPLAY   - go back to the checkpoint after `merge` and run from there
4. FORK     - edit the merged findings at that checkpoint, continue as a new branch

Needs research_agent_parallel.py in the same folder/notebook session.
Notebook: init_llm(); demo("your question")
"""


from research_agent_parallel import build_graph, init_llm  # noqa: F401
from pathlib import Path
project_dir = Path(r"C:\Users\asadu\Desktop\Ai_practice")
print("Folder hai:", project_dir.exists())
print([p.name for p in project_dir.glob("*.py")])   # folder ki .py files
print("Notebook yahan chal rahi hai:", Path.cwd())
from langgraph.types import Command

def summarize(values: dict) -> str:
    """One-line view of what the state contains at a checkpoint."""
    return (
        f"tasks={len(values.get('tasks', []))} "
        f"results={len(values.get('results', []))} "
        f"merged={'yes' if values.get('merged') else 'no'} "
        f"report={'yes' if values.get('report') else 'no'}"
    )


def short_id(config: dict) -> str:
    return config["configurable"]["checkpoint_id"][:13]


def show_history(app, config) -> list:
    """INSPECT: print all checkpoints oldest -> newest and return them."""
    history = list(reversed(list(app.get_state_history(config))))  # oldest first
    print(f"\n{'#':>2}  {'checkpoint':<14} {'parent':<14} {'next node(s)':<28} state")
    for i, snap in enumerate(history):
        cid = short_id(snap.config)
        parent = short_id(snap.parent_config) if snap.parent_config else "-"
        nxt = ", ".join(snap.next) or "(done)"
        print(f"{i:>2}  {cid:<14} {parent:<14} {nxt:<28} {summarize(snap.values)}")
    return history


def show_graph(app, filename: str = "graph.png") -> None:
    """Print the workflow graph: image in a notebook (and saved to a file),
    Mermaid text as a fallback (paste it into mermaid.live)."""
    graph = app.get_graph()
    try:
        png = graph.draw_mermaid_png()          # needs internet (mermaid.ink)
        with open(filename, "wb") as f:
            f.write(png)
        print(f"Graph image saved to {filename}")
        try:
            from IPython.display import Image, display
            display(Image(png))
        except ImportError:
            pass
    except Exception as e:
        print("Graph image not generated:", e)
        print("Paste this into https://mermaid.live :\n")
        print(graph.draw_mermaid())


def run_and_approve(app, graph_input, config, approve: bool = True) -> dict:
    """Run; if the graph pauses at human_review, answer the interrupt."""
    out = app.invoke(graph_input, config)
    if "__interrupt__" in out:
        out = app.invoke(Command(resume=approve), config)
    return out


def demo(question: str = "How are companies using AI agents in customer support?",
         thread_id: str = "time-travel-1"):
    app = build_graph()
    config = {"configurable": {"thread_id": thread_id}, "max_concurrency": 2}

    # 1) normal run
    print("=== 1) Normal run ===")
    out = run_and_approve(app, {"question": question, "results": []}, config)
    print("Report v1:\n", out["report"][:400], "...\n")

    # 2) inspect
    print("=== 2) INSPECT: checkpoints of this thread ===")
    history = show_history(app, config)

    # the checkpoint right after `merge`: next node is human_review
    after_merge = next(s for s in history if s.next == ("human_review",))
    print("\nGoing back to checkpoint", short_id(after_merge.config),
          "(after merge, before human_review)")

    # 3) replay: planner/workers/merge are NOT run again, only what comes after
    print("\n=== 3) REPLAY from that checkpoint ===")
    out = app.invoke(None, after_merge.config)            # re-runs human_review (pauses again)
    if "__interrupt__" in out:
        out = app.invoke(Command(resume=True), config)    # approve -> writer runs again
    print("Report v2 (same findings, writer ran again):\n", out["report"][:400], "...\n")

    # 4) fork: change the state at that checkpoint, continue as a new branch
    print("=== 4) FORK: edit merged findings, continue ===")
    edited = after_merge.values["merged"] + "\n\n## Editor note\nFocus the report on cost savings."
    fork_config = app.update_state(after_merge.config, {"merged": edited}, as_node="merge")
    out = app.invoke(None, fork_config)                   # human_review pauses again
    if "__interrupt__" in out:
        out = app.invoke(Command(resume=True), config)    # resume the newest branch
    print("Report v3 (forked, edited findings):\n", out["report"][:400], "...\n")

    print("=== Checkpoints after replay + fork ===")
    show_history(app, config)

    print("\n=== Workflow graph ===")
    show_graph(app)
    return app


if __name__ == "__main__":
    init_llm()
    demo()

Folder hai: True
[]
Notebook yahan chal rahi hai: c:\Users\asadu\Desktop\Ai_practice


NameError: name 'init_llm' is not defined